# <font color="#418FDE" size="6.5" uppercase>**Training Networks**</font>

>Last update: 20260708.
    
By the end of this Lecture, you will be able to:
- Explain gradient descent, stochastic gradient descent, and backpropagation at an intuitive level. 
- Implement a small neural network training loop from scratch using NumPy. 
- Evaluate neural network performance and identify signs of overfitting. 


## **1. Learning Mechanics**

### **1.1. Gradient Descent Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_01_01.jpg?v=1783566702" width="250">



>* Networks learn by adjusting weights and biases
>* Gradient descent steps downhill to reduce loss

>* Gradients show which weights need adjustment
>* Learning rate controls stable training steps

>* Learning improves through repeated small updates
>* Good enough solutions may replace perfection



In [ ]:
#@title Python Code - Gradient Descent Basics

# Gradient descent updates one parameter gradually.
# This example uses a simple civil dataset.
# The plot shows downhill learning behavior.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic beam deflection training data.
np.random.seed(7)
loads_kips = np.array([1, 2, 3, 4, 5, 6], dtype=float)
true_deflection_inches = 0.18 * loads_kips + 0.05

# Add small measurement noise for realism.
noise_inches = np.random.normal(0.0, 0.015, loads_kips.shape)
measured_inches = true_deflection_inches + noise_inches

# Validate matching one dimensional arrays.
assert loads_kips.ndim == measured_inches.ndim == 1
assert loads_kips.size == measured_inches.size

# Start with a poor slope estimate.
slope = 0.02
learning_rate = 0.01
loss_history = []

# Repeatedly step opposite the loss gradient.
for step in range(60):
    predictions = slope * loads_kips
    errors = predictions - measured_inches
    loss = np.mean(errors ** 2)

    gradient = 2.0 * np.mean(errors * loads_kips)
    slope = slope - learning_rate * gradient
    loss_history.append(loss)

# Evaluate the final fitted line.
final_predictions = slope * loads_kips
final_loss = np.mean((final_predictions - measured_inches) ** 2)

# Print only a few teaching summary lines.
print(f"NumPy version: {np.__version__}")
print(f"Initial slope guess: 0.020 inches per kip")
print(f"Learned slope: {slope:.3f} inches per kip")
print(f"Final mean squared error: {final_loss:.5f}")
print("Gradient descent repeatedly takes small downhill steps.")

# Plot loss decreasing across update steps.
plt.figure(figsize=(7, 4))
plt.plot(loss_history, marker="o", markersize=3)
plt.xlabel("Gradient descent step")
plt.ylabel("Mean squared error")
plt.title("Loss decreases as the parameter improves")
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Stochastic Gradient Descent**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_01_02.jpg?v=1783566704" width="250">



>* Updates weights using small random data batches
>* Noisy quick steps reduce overall error

>* Small batches make training faster
>* Frequent updates scale to huge datasets

>* Random updates can escape poor solutions
>* Learning rate controls noisy training progress



In [ ]:
#@title Python Code - Stochastic Gradient Descent

# This example compares batch and stochastic updates.
# A tiny bridge dataset keeps training understandable.
# NumPy shows the mechanics without ML libraries.

import numpy as np
import matplotlib.pyplot as plt

# Set randomness for repeatable stochastic batches.
rng = np.random.default_rng(7)

# Create small civil engineering style measurements.
loads = np.linspace(10.0, 80.0, 40)
true_deflection = 0.08 * loads + 0.6
noise = rng.normal(0.0, 0.35, loads.size)

# Deflection is measured in millimeters here.
deflections = true_deflection + noise
x_values = (loads - loads.mean()) / loads.std()
y_values = deflections

# Validate matching one dimensional training arrays.
assert x_values.shape == y_values.shape
assert x_values.ndim == 1

# Define mean squared error for one line model.
def mse_loss(weight, bias, x_batch, y_batch):
    predictions = weight * x_batch + bias
    errors = predictions - y_batch
    return np.mean(errors ** 2)

# Train using either full batch or mini batches.
def train_model(batch_size, epochs, learning_rate):
    weight = 0.0
    bias = 0.0
    losses = []

    for epoch in range(epochs):
        order = rng.permutation(x_values.size)
        for start in range(0, x_values.size, batch_size):
            batch_ids = order[start:start + batch_size]
            xb = x_values[batch_ids]
            yb = y_values[batch_ids]

            predictions = weight * xb + bias
            errors = predictions - yb
            grad_w = 2.0 * np.mean(errors * xb)
            grad_b = 2.0 * np.mean(errors)

            weight -= learning_rate * grad_w
            bias -= learning_rate * grad_b
        losses.append(mse_loss(weight, bias, x_values, y_values))

    return weight, bias, np.array(losses)

# Full batch uses every example before updating.
full_w, full_b, full_losses = train_model(40, 35, 0.08)

# Stochastic mini batch updates after small samples.
sgd_w, sgd_b, sgd_losses = train_model(4, 35, 0.08)

# Print only a compact teaching summary.
print(f"NumPy version: {np.__version__}")
print(f"Full batch final loss: {full_losses[-1]:.3f}")
print(f"Mini-batch SGD final loss: {sgd_losses[-1]:.3f}")
print("SGD loss may wiggle, but usually trends downward.")

# Plot both learning paths for visual comparison.
plt.figure(figsize=(7, 4))
plt.plot(full_losses, label="Full batch gradient descent")
plt.plot(sgd_losses, label="Mini-batch stochastic gradient descent")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Noisy but useful SGD learning steps")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **1.3. Backpropagation Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_01_03.jpg?v=1783566706" width="250">



>* Forward pass measures prediction error
>* Backward pass assigns blame to parameters

>* Layered operations shape each prediction
>* Backward error signals guide targeted updates

>* Backpropagation guides optimizers toward lower error
>* Stable learning depends on good feedback signals



In [ ]:
#@title Python Code - Backpropagation Basics

# Backpropagation traces error signals backward.
# This tiny network predicts beam safety.
# NumPy shows every learning step.

# NumPy and matplotlib are preinstalled in Colab.
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable teaching results.
rng = np.random.default_rng(7)

# Create tiny civil engineering style training data.
span_ft = np.array([10, 12, 14, 16, 18, 20], dtype=float)
load_kips = np.array([4, 5, 7, 8, 10, 12], dtype=float)

# Normalize inputs to keep gradients well behaved.
X = np.column_stack((span_ft / 20, load_kips / 12))
y = np.array([[0], [0], [0], [1], [1], [1]], dtype=float)

# Validate shapes before matrix operations begin.
assert X.shape == (6, 2) and y.shape == (6, 1)

# Initialize a small two layer neural network.
W1 = rng.normal(0, 0.4, size=(2, 3))
b1 = np.zeros((1, 3))
W2 = rng.normal(0, 0.4, size=(3, 1))
b2 = np.zeros((1, 1))

# Define sigmoid activation for probabilities.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Store loss values for one compact plot.
loss_history = []
learning_rate = 0.8

# Train silently using full batch gradient descent.
for epoch in range(400):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    yhat = sigmoid(z2)

    # Compute mean squared error loss.
    error = yhat - y
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    # Backpropagate output layer responsibility.
    d_z2 = error * yhat * (1 - yhat)
    d_W2 = a1.T @ d_z2 / len(X)
    d_b2 = np.mean(d_z2, axis=0, keepdims=True)

    # Backpropagate hidden layer responsibility.
    d_a1 = d_z2 @ W2.T
    d_z1 = d_a1 * a1 * (1 - a1)
    d_W1 = X.T @ d_z1 / len(X)
    d_b1 = np.mean(d_z1, axis=0, keepdims=True)

    # Gradient descent updates each parameter slightly.
    W2 -= learning_rate * d_W2
    b2 -= learning_rate * d_b2
    W1 -= learning_rate * d_W1
    b1 -= learning_rate * d_b1

# Evaluate final predictions after learning.
final_probs = sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2)
final_classes = (final_probs >= 0.5).astype(int)
accuracy = np.mean(final_classes == y)

# Print only concise teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {loss_history[0]:.3f}")
print(f"Final loss: {loss_history[-1]:.3f}")
print(f"Training accuracy: {accuracy:.0%}")
print("Backpropagation computed gradients before each update.")

# Plot loss curve to show learning progress.
plt.figure(figsize=(6, 4))
plt.plot(loss_history, color="steelblue")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Backpropagation guides gradient descent")
plt.grid(True, alpha=0.3)
plt.show()



## **2. Training Code**

### **2.1. NumPy Training Loop**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_02_01.jpg?v=1783566697" width="250">



>* Loop predicts, measures errors, updates parameters
>* NumPy makes each training step explicit

>* Train with batches, forward pass, and loss
>* Backpropagate gradients and update parameters carefully

>* Track loss and accuracy after each epoch
>* Use trends to diagnose learning problems



In [ ]:
#@title Python Code - NumPy Training Loop

# Build a tiny neural network manually.
# Track loss and validation accuracy.
# Use NumPy for every training step.

# NumPy is already available in Colab.
import numpy as np
import matplotlib.pyplot as plt

# Print the numerical library version.
print("NumPy version:", np.__version__)

# Create reproducible synthetic civil engineering data.
rng = np.random.default_rng(7)
samples = 240

# Generate two simple bridge inspection features.
crack_mm = rng.normal(2.5, 1.0, samples)
span_ft = rng.normal(80.0, 18.0, samples)

# Combine features into one design matrix.
X_raw = np.column_stack((crack_mm, span_ft))
assert X_raw.shape == (samples, 2)

# Create labels from a hidden risk rule.
score = 1.4 * crack_mm + 0.025 * span_ft
score += rng.normal(0.0, 0.6, samples)

# Convert scores into binary repair labels.
y = (score > 5.2).astype(float).reshape(-1, 1)
assert y.shape == (samples, 1)

# Shuffle before splitting train and validation sets.
order = rng.permutation(samples)
X_raw = X_raw[order]
y = y[order]

# Split data into training and validation portions.
train_n = 180
X_train_raw = X_raw[:train_n]
X_val_raw = X_raw[train_n:]

# Split labels using the same boundary.
y_train = y[:train_n]
y_val = y[train_n:]

# Standardize features using training statistics only.
mean = X_train_raw.mean(axis=0, keepdims=True)
std = X_train_raw.std(axis=0, keepdims=True) + 1e-8

# Apply the same scaling to both datasets.
X_train = (X_train_raw - mean) / std
X_val = (X_val_raw - mean) / std

# Define a stable sigmoid activation function.
def sigmoid(z):
    z = np.clip(z, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-z))

# Define binary cross entropy loss.
def binary_loss(pred, target):
    eps = 1e-8
    pred = np.clip(pred, eps, 1.0 - eps)
    return -np.mean(target * np.log(pred) + (1 - target) * np.log(1 - pred))

# Initialize a small two layer network.
hidden = 5
W1 = rng.normal(0.0, 0.3, (2, hidden))
b1 = np.zeros((1, hidden))

# Initialize output layer parameters.
W2 = rng.normal(0.0, 0.3, (hidden, 1))
b2 = np.zeros((1, 1))

# Set compact training loop settings.
epochs = 120
batch_size = 30
learning_rate = 0.08

# Store history for one simple plot.
train_losses = []
val_losses = []

# Train with mini batch gradient descent.
for epoch in range(epochs):
    order = rng.permutation(train_n)
    X_epoch = X_train[order]
    y_epoch = y_train[order]

    # Process one mini batch at a time.
    for start in range(0, train_n, batch_size):
        end = start + batch_size
        xb = X_epoch[start:end]
        yb = y_epoch[start:end]

        # Forward pass computes predictions.
        z1 = xb @ W1 + b1
        a1 = np.tanh(z1)
        pred = sigmoid(a1 @ W2 + b2)

        # Backpropagation computes parameter gradients.
        m = xb.shape[0]
        dz2 = (pred - yb) / m
        dW2 = a1.T @ dz2
        db2 = dz2.sum(axis=0, keepdims=True)

        # Continue gradients through hidden layer.
        da1 = dz2 @ W2.T
        dz1 = da1 * (1.0 - a1 * a1)
        dW1 = xb.T @ dz1
        db1 = dz1.sum(axis=0, keepdims=True)

        # Gradient descent updates every parameter.
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    # Measure progress after each epoch.
    train_pred = sigmoid(np.tanh(X_train @ W1 + b1) @ W2 + b2)
    val_pred = sigmoid(np.tanh(X_val @ W1 + b1) @ W2 + b2)

    # Save losses for diagnosis.
    train_losses.append(binary_loss(train_pred, y_train))
    val_losses.append(binary_loss(val_pred, y_val))

# Convert probabilities into class predictions.
train_class = (train_pred >= 0.5).astype(float)
val_class = (val_pred >= 0.5).astype(float)

# Compute simple accuracy values.
train_acc = np.mean(train_class == y_train)
val_acc = np.mean(val_class == y_val)

# Print a compact training summary.
print("Final train loss:", round(train_losses[-1], 3))
print("Final validation loss:", round(val_losses[-1], 3))
print("Train accuracy:", round(float(train_acc), 3))
print("Validation accuracy:", round(float(val_acc), 3))

# Give a short overfitting diagnostic.
gap = train_losses[-1] - val_losses[-1]
message = "No strong overfitting signal" if gap < 0.05 else "Check validation behavior"
print("Diagnostic:", message)

# Plot loss curves as the instrument panel.
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross entropy")
plt.title("NumPy neural network training loop")
plt.legend()
plt.tight_layout()
plt.show()



### **2.2. Loss Computation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_02_02.jpg?v=1783566698" width="250">



>* Loss measures prediction error for each task
>* Loss gradients guide weight updates

>* Match prediction and target array shapes
>* Average batch losses into one scalar

>* Keep loss calculations numerically stable
>* Use loss trends with validation context



In [ ]:
#@title Python Code - Loss Computation

# This example computes neural network losses.
# Tiny arrays keep every calculation visible.
# Civil labels mimic simple damage classification.

# Import NumPy for vectorized array calculations.
import numpy as np

# Set print formatting for readable probabilities.
np.set_printoptions(precision=3, suppress=True)

# Store raw network scores for three inspections.
logits = np.array([[2.0, 0.5], [0.2, 1.8], [1.2, 1.0]])

# Store correct class labels for examples.
y_true = np.array([0, 1, 1])

# Validate batch and label shapes before computing.
assert logits.shape[0] == y_true.shape[0]
assert logits.ndim == 2 and y_true.ndim == 1

# Shift logits to improve softmax stability.
shifted = logits - np.max(logits, axis=1, keepdims=True)

# Convert shifted scores into class probabilities.
exp_scores = np.exp(shifted)
probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Clip probabilities before taking logarithms safely.
epsilon = 1e-12
safe_probs = np.clip(probabilities, epsilon, 1.0)

# Select probabilities assigned to correct classes.
row_ids = np.arange(y_true.size)
correct_probs = safe_probs[row_ids, y_true]

# Compute per example cross entropy losses.
example_losses = -np.log(correct_probs)
mean_loss = np.mean(example_losses)

# Compute simple accuracy for interpretation.
predicted_labels = np.argmax(probabilities, axis=1)
accuracy = np.mean(predicted_labels == y_true)

# Print compact teaching results only.
print("NumPy version:", np.__version__)
print("Predicted probabilities:\n", probabilities)
print("Per-example losses:", np.round(example_losses, 3))
print("Mean batch loss:", round(float(mean_loss), 3))
print("Batch accuracy:", round(float(accuracy), 3))



### **2.3. Loss Computation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_02_03.jpg?v=1783566700" width="250">



>* Loss measures prediction error for each task
>* Reducing loss guides weight updates

>* Match prediction and target array shapes
>* Use true-class confidence for smooth learning

>* Average batch loss for stable tracking
>* Use loss trends to diagnose training



In [ ]:
#@title Python Code - Loss Computation

# This example computes losses for tiny batches.
# Civil engineering predictions use regression and classification.
# NumPy shows each loss calculation clearly.

import numpy as np

# Set print formatting for readable numbers.
np.set_printoptions(precision=4, suppress=True)

# Create tiny regression targets and predictions.
y_true_depth = np.array([[12.0], [18.0], [25.0]])
y_pred_depth = np.array([[10.5], [20.0], [24.0]])

# Validate matching regression array shapes.
assert y_true_depth.shape == y_pred_depth.shape

# Compute mean squared error across batch.
squared_errors = (y_pred_depth - y_true_depth) ** 2
mse_loss = np.mean(squared_errors)

# Create class labels for pavement condition.
y_true_class = np.array([0, 2, 1])
logits = np.array([[2.0, 0.5, -1.0], [0.1, 0.2, 1.8], [0.3, 1.2, 0.4]])

# Validate classification batch dimensions.
assert logits.shape[0] == y_true_class.size
assert logits.shape[1] == 3

# Convert raw scores into stable probabilities.
shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
exp_scores = np.exp(shifted_logits)
probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Select probabilities assigned to correct classes.
batch_indices = np.arange(y_true_class.size)
correct_probs = probabilities[batch_indices, y_true_class]

# Compute average cross entropy loss safely.
epsilon = 1e-12
cross_entropy = -np.mean(np.log(correct_probs + epsilon))

# Print compact teaching results only.
print(f"Regression MSE loss: {mse_loss:.3f} square inches")
print(f"Classification cross-entropy loss: {cross_entropy:.3f}")
print(f"Correct-class probabilities: {correct_probs}")
print("Lower loss means predictions better match targets.")



## **3. Network Evaluation**

### **3.1. Test Performance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_03_01.jpg?v=1783566691" width="250">



>* Test sets reveal real generalization
>* Training success may hide memorization

>* Choose metrics that match the task
>* Check errors, imbalance, and real-world goals

>* Use realistic, separate test data
>* Compare splits to spot weak generalization



In [ ]:
#@title Python Code - Test Performance

# Test performance estimates real deployment behavior.
# Separate data reveals memorization and overfitting.
# This example uses synthetic bridge inspections.

# NumPy and matplotlib are usually preinstalled.
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(7)

# Create small civil engineering inspection data.
n_samples = 240
age_years = rng.uniform(1, 80, n_samples)
crack_mm = rng.uniform(0, 12, n_samples)

# Combine features into a simple matrix.
X = np.column_stack((age_years, crack_mm))
assert X.shape == (n_samples, 2)

# Define risk labels with controlled noise.
score = 0.055 * age_years + 0.45 * crack_mm
score += rng.normal(0, 0.9, n_samples)
y = (score > 5.2).astype(float).reshape(-1, 1)

# Shuffle before splitting into datasets.
order = rng.permutation(n_samples)
X = X[order]
y = y[order]

# Split into training, validation, and test sets.
train_end = 140
valid_end = 190
X_train, y_train = X[:train_end], y[:train_end]
X_valid, y_valid = X[train_end:valid_end], y[train_end:valid_end]
X_test, y_test = X[valid_end:], y[valid_end:]

# Standardize using training statistics only.
mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_valid = (X_valid - mean) / std
X_test = (X_test - mean) / std

# Add bias columns for logistic regression.
ones_train = np.ones((X_train.shape[0], 1))
ones_valid = np.ones((X_valid.shape[0], 1))
ones_test = np.ones((X_test.shape[0], 1))

# Build design matrices with intercept terms.
X_train_b = np.hstack((ones_train, X_train))
X_valid_b = np.hstack((ones_valid, X_valid))
X_test_b = np.hstack((ones_test, X_test))

# Define a stable sigmoid function.
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1 / (1 + np.exp(-z))

# Train a small model silently.
w = np.zeros((X_train_b.shape[1], 1))
learning_rate = 0.25
for epoch in range(350):
    predictions = sigmoid(X_train_b @ w)
    gradient = X_train_b.T @ (predictions - y_train)
    w -= learning_rate * gradient / len(y_train)

# Define accuracy for any split.
def accuracy(Xb, labels):
    probabilities = sigmoid(Xb @ w)
    predicted = (probabilities >= 0.5).astype(float)
    return float(np.mean(predicted == labels))

# Evaluate training, validation, and test performance.
train_acc = accuracy(X_train_b, y_train)
valid_acc = accuracy(X_valid_b, y_valid)
test_acc = accuracy(X_test_b, y_test)
gap = train_acc - test_acc

# Print concise evaluation results.
print(f"NumPy version: {np.__version__}")
print(f"Training accuracy:   {train_acc:.2f}")
print(f"Validation accuracy: {valid_acc:.2f}")
print(f"Test accuracy:       {test_acc:.2f}")
print(f"Train-test gap:      {gap:.2f}")

# Interpret the generalization gap simply.
if gap > 0.15:
    message = "Warning: possible overfitting."
else:
    message = "Gap is modest; generalization looks reasonable."
print(message)

# Plot split accuracies for visual comparison.
labels = ["Train", "Validation", "Test"]
values = [train_acc, valid_acc, test_acc]
plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=["steelblue", "orange", "seagreen"])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Network Evaluation: Test Performance")
plt.show()



### **3.2. Overfitting Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_03_02.jpg?v=1783566693" width="250">



>* Overfitting memorizes training quirks, not general patterns
>* Check performance on unseen validation or test data

>* Validation worsens while training keeps improving
>* Model memorizes rare training details

>* Watch for unstable, overconfident new-data predictions
>* Check errors, subgroups, and real-world consistency



In [ ]:
#@title Python Code - Overfitting Signs

# This example shows common overfitting signs.
# Training improves while validation performance worsens.
# Civil data can show similar behavior.

# numpy and matplotlib are usually preinstalled.

# Import small numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Print the NumPy version briefly.
print("NumPy version:", np.__version__)

# Make the random example reproducible.
rng = np.random.default_rng(7)

# Create a small noisy civil engineering dataset.
x_train = np.linspace(0, 10, 18)
y_train = 2.5 * x_train + rng.normal(0, 3.0, 18)

# Create validation data from the same broad pattern.
x_val = np.linspace(0.5, 9.5, 18)
y_val = 2.5 * x_val + rng.normal(0, 3.0, 18)

# Validate matching input and target lengths.
assert x_train.shape == y_train.shape
assert x_val.shape == y_val.shape

# Fit a simple line and an overly flexible curve.
line_model = np.polyfit(x_train, y_train, 1)
curve_model = np.polyfit(x_train, y_train, 12)

# Define mean squared error for evaluation.
def mse(poly, x_values, y_values):
    predictions = np.polyval(poly, x_values)
    return np.mean((predictions - y_values) ** 2)

# Compute training and validation errors.
line_train = mse(line_model, x_train, y_train)
line_val = mse(line_model, x_val, y_val)
curve_train = mse(curve_model, x_train, y_train)
curve_val = mse(curve_model, x_val, y_val)

# Print compact evaluation results.
print("Simple line train MSE:", round(line_train, 2))
print("Simple line validation MSE:", round(line_val, 2))
print("Flexible curve train MSE:", round(curve_train, 2))
print("Flexible curve validation MSE:", round(curve_val, 2))

# Explain the overfitting warning sign.
print("Warning: low training error with high validation error suggests overfitting.")

# Prepare smooth x values for plotting.
x_grid = np.linspace(0, 10, 200)
line_pred = np.polyval(line_model, x_grid)
curve_pred = np.polyval(curve_model, x_grid)

# Plot data and both fitted models.
plt.figure(figsize=(7, 4))
plt.scatter(x_train, y_train, label="training data")
plt.scatter(x_val, y_val, label="validation data")
plt.plot(x_grid, line_pred, label="simple line")
plt.plot(x_grid, curve_pred, label="flexible curve")

# Add labels for civil engineering context.
plt.xlabel("Beam span, meters")
plt.ylabel("Measured deflection, millimeters")
plt.title("Overfitting sign: validation performance gap")
plt.legend()

# Keep the plot readable in Colab.
plt.ylim(-10, 40)
plt.grid(True, alpha=0.3)
plt.show()



### **3.3. Overfitting Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_03_03.jpg?v=1783566695" width="250">



>* Overfitting memorizes training quirks, not general patterns
>* Validate performance on unseen data

>* Training improves while validation worsens
>* Validation warns of poor real-world generalization

>* Unstable confidence signals weak generalization
>* Compare datasets and group errors



In [ ]:
#@title Python Code - Overfitting Signs

# This example shows overfitting signs.
# Training and validation curves are compared.
# A tiny network uses NumPy only.

import numpy as np
import matplotlib.pyplot as plt

# Print the numerical library version.
print("NumPy version:", np.__version__)

# Make results repeatable for learners.
rng = np.random.default_rng(7)

# Create a small noisy engineering dataset.
n_train, n_val = 24, 120
x_train = np.linspace(-3, 3, n_train).reshape(-1, 1)
x_val = np.linspace(-3, 3, n_val).reshape(-1, 1)

# Simulate settlement measurements in centimeters.
y_train = np.sin(x_train) + 0.25 * rng.normal(size=x_train.shape)
y_val = np.sin(x_val)

# Validate the expected array shapes.
assert x_train.shape == y_train.shape
assert x_val.shape == y_val.shape

# Initialize an intentionally flexible network.
hidden = 40
W1 = rng.normal(0, 0.7, size=(1, hidden))
b1 = np.zeros((1, hidden))

# Initialize the output layer parameters.
W2 = rng.normal(0, 0.7, size=(hidden, 1))
b2 = np.zeros((1, 1))

# Store losses for curve inspection.
train_losses, val_losses = [], []
lr, epochs = 0.03, 900

# Train silently using full batch gradient descent.
for epoch in range(epochs):
    z1 = x_train @ W1 + b1
    a1 = np.tanh(z1)

    y_hat = a1 @ W2 + b2
    error = y_hat - y_train
    loss = np.mean(error ** 2)

    d_yhat = 2 * error / n_train
    dW2 = a1.T @ d_yhat
    db2 = np.sum(d_yhat, axis=0, keepdims=True)

    da1 = d_yhat @ W2.T
    dz1 = da1 * (1 - a1 ** 2)
    dW1 = x_train.T @ dz1

    db1 = np.sum(dz1, axis=0, keepdims=True)
    W2 -= lr * dW2
    b2 -= lr * db2

    W1 -= lr * dW1
    b1 -= lr * db1
    if epoch % 10 == 0:
        train_losses.append(loss)

        val_pred = np.tanh(x_val @ W1 + b1) @ W2 + b2
        val_losses.append(np.mean((val_pred - y_val) ** 2))

# Summarize the overfitting warning signs.
best_val = min(val_losses)
final_gap = val_losses[-1] - train_losses[-1]
print("Final training loss:", round(train_losses[-1], 4))

print("Final validation loss:", round(val_losses[-1], 4))
print("Best validation loss:", round(best_val, 4))
print("Final validation-training gap:", round(final_gap, 4))

# Plot both curves to reveal divergence.
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")

plt.xlabel("Checkpoint every 10 epochs")
plt.ylabel("Mean squared error")
plt.title("Overfitting sign: validation loss stops improving")
plt.legend()

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Training Networks**</font>


In this lecture, you learned to:
- Explain gradient descent, stochastic gradient descent, and backpropagation at an intuitive level. 
- Implement a small neural network training loop from scratch using NumPy. 
- Evaluate neural network performance and identify signs of overfitting. 

In the next Module (Module 8), we will go over 'None'